# 🎯 Sony Bravia TV Reviews — NLP & Neural Network Classifier

**Objective:** Build a model that can analyze and classify product reviews using Natural Language Processing (NLP) and Neural Networks.

**Dataset:** `SonyBravia_TV_Reviews.csv` — ~5,000 Sony Bravia TV product reviews.

**Classification Task:** 3-class sentiment classification
- `0` → **Negative** (1-2 stars equivalent)
- `1` → **Neutral**  (3 stars equivalent)
- `2` → **Positive** (4-5 stars equivalent)

---
### Project Pipeline
```
Raw Reviews → EDA → Text Cleaning → Labeling → Vectorization
→ Dense NN → LSTM → 1D-CNN → BiLSTM+Attention
→ Evaluation → Optimization → Error Analysis → LIME Explainability
```

## 📦 Section 0: Setup & Imports

Import all necessary libraries for data processing, visualisation, NLP, and neural network construction.

In [ ]:
pip install pandas numpy lime matplotlib seaborn nltk tensorflow

In [ ]:
# ─── Standard libraries ──────────────────────────────────────────────────────
import os
import re
import string
import warnings
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter

warnings.filterwarnings('ignore')

# ─── NLP libraries ───────────────────────────────────────────────────────────
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords',    quiet=True)
nltk.download('wordnet',      quiet=True)
nltk.download('punkt',        quiet=True)
nltk.download('omw-1.4',      quiet=True)

# ─── TensorFlow / Keras ───────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

# ─── Scikit-learn metrics ────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score
)

# ─── Reproducibility seed ────────────────────────────────────────────────────
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ─── Global hyper-parameters ────────────────────────────────────────────────
MAX_WORDS   = 10000   # vocabulary size
MAX_LEN     = 60      # max sequence length (tokens)
EMBED_DIM   = 100     # GloVe embedding dimension
BATCH_SIZE  = 64
EPOCHS      = 15
NUM_CLASSES = 3       # Negative / Neutral / Positive

print(f"TensorFlow version : {tf.__version__}")
print(f"Keras version      : {keras.__version__}")
print(f"NumPy version      : {np.__version__}")
print(f"Pandas version     : {pd.__version__}")
print("✅ All libraries loaded successfully!")


## 📊 Section 1: Data Loading & Exploratory Data Analysis (EDA)

Understand the structure of the dataset before building any model.

**Key EDA steps:**
1. Load the CSV and inspect shape, dtypes, and missing values
2. Review-length distribution
3. Word-frequency analysis (top-40 most common words)
4. Class distribution (after labeling is done in Section 3)

In [ ]:
# ─── 1.1  Load the raw CSV ────────────────────────────────────────────────────
df_raw = pd.read_csv('SonyBravia_TV_Reviews.csv')

print("Shape:", df_raw.shape)
print("\nColumn names:", df_raw.columns.tolist())
print("\nData types:\n", df_raw.dtypes)
print("\nMissing values:\n", df_raw.isnull().sum())
df_raw.head(10)


In [ ]:
# ─── 1.2  Basic statistics on review text ────────────────────────────────────
df_raw['review_length'] = df_raw['Review Text'].astype(str).apply(lambda x: len(x.split()))
df_raw['char_length']   = df_raw['Review Text'].astype(str).apply(len)

print("Review word-count statistics:")
print(df_raw['review_length'].describe().round(2))
print("\nCharacter-count statistics:")
print(df_raw['char_length'].describe().round(2))


In [ ]:
# ─── 1.3  Review-length distribution ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df_raw['review_length'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Review Word Count', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Words')
axes[0].set_ylabel('Frequency')

axes[1].hist(df_raw['char_length'], bins=30, color='coral', edgecolor='white')
axes[1].set_title('Distribution of Review Character Count', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of Characters')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('eda_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Length distribution plots saved.")


In [ ]:
# ─── 1.4  Word-frequency analysis ────────────────────────────────────────────
all_words = ' '.join(df_raw['Review Text'].astype(str).tolist()).lower().split()
word_freq  = Counter(all_words).most_common(40)
words_  = [w[0] for w in word_freq]
counts_ = [w[1] for w in word_freq]

plt.figure(figsize=(14, 5))
bars = plt.bar(words_, counts_, color=plt.cm.Blues_r(np.linspace(0.3, 0.9, 40)))
plt.xticks(rotation=70, ha='right', fontsize=9)
plt.title('Top 40 Most Frequent Words (Before Cleaning)', fontsize=13, fontweight='bold')
plt.xlabel('Word')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('eda_word_frequency.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Word frequency plot saved.")


In [ ]:
# ─── 1.5  How many unique products are in the dataset? ───────────────────────
print("Unique product names:", df_raw['Product Name'].nunique())
print(df_raw['Product Name'].value_counts())


## 🧹 Section 2: Text Cleaning & Preprocessing

A clean text representation leads to better NLP performance.

**Steps performed:**
1. Remove `Amazon -` prefix (review source tag)
2. Lowercase all text
3. Remove HTML tags (regex)
4. Remove punctuation and special characters
5. Strip extra whitespace
6. Remove English stopwords (NLTK)
7. Lemmatize words (WordNetLemmatizer)

In [ ]:
# ─── 2.1  Define the cleaning pipeline ───────────────────────────────────────
lemmatizer = WordNetLemmatizer()
STOP_WORDS  = set(stopwords.words('english'))

def clean_text(text: str) -> str:
    """Full text-cleaning pipeline."""
    # (a) Remove 'Amazon -' source prefix
    text = re.sub(r'(?i)^Amazon\s*-\s*', '', text)

    # (b) Lowercase
    text = text.lower()

    # (c) Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    # (d) Remove URLs
    text = re.sub(r'http\S+|www\.\S+', ' ', text)

    # (e) Remove punctuation and special characters
    text = re.sub(r"[^a-z\s]", ' ', text)

    # (f) Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # (g) Tokenize and remove stopwords
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 2]

    # (h) Lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens]

    return ' '.join(tokens)

# Apply cleaning
df_raw['clean_text'] = df_raw['Review Text'].astype(str).apply(clean_text)

print("Sample Before Cleaning:")
print(df_raw['Review Text'].iloc[0])
print("\nSample After Cleaning:")
print(df_raw['clean_text'].iloc[0])


In [ ]:
# ─── 2.2  Drop rows where cleaning resulted in empty text ────────────────────
df_raw = df_raw[df_raw['clean_text'].str.strip().str.len() > 0].copy()
df_raw.reset_index(drop=True, inplace=True)

print(f"Rows after cleaning: {len(df_raw)}")
assert len(df_raw) > 4000, "Too many rows dropped during cleaning!"

# Update review length after cleaning
df_raw['clean_length'] = df_raw['clean_text'].apply(lambda x: len(x.split()))
print("\nCleaned review word-count stats:")
print(df_raw['clean_length'].describe().round(2))


In [ ]:
# ─── 2.3  Word frequency AFTER cleaning (top 40) ────────────────────────────
all_clean_words = ' '.join(df_raw['clean_text'].tolist()).split()
clean_freq      = Counter(all_clean_words).most_common(40)
cwords_  = [w[0] for w in clean_freq]
ccounts_ = [w[1] for w in clean_freq]

plt.figure(figsize=(14, 5))
plt.bar(cwords_, ccounts_, color=plt.cm.Greens_r(np.linspace(0.3, 0.9, 40)))
plt.xticks(rotation=70, ha='right', fontsize=9)
plt.title('Top 40 Most Frequent Words (After Cleaning)', fontsize=13, fontweight='bold')
plt.xlabel('Word')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('eda_clean_word_frequency.png', dpi=150, bbox_inches='tight')
plt.show()


## 🏷️ Section 3: Sentiment Labeling

Since the dataset lacks star-ratings, we create a **custom 3-class label** using a lexicon-based keyword scorer.

**Scoring logic:**
- Each positive signal word adds `+1` to the score
- Each negative signal word adds `-1` to the score
- `score ≥ 2` → **Positive (2)**
- `score == 1` → **Positive (2)** (lean positive)
- `score == 0` → **Neutral (1)**
- `score < 0`  → **Negative (0)**

This matches the bonus idea: **Negative [1-2★], Neutral [3★], Positive [4-5★]**

In [ ]:
# ─── 3.1  Define sentiment lexicons ──────────────────────────────────────────
POSITIVE_WORDS = {
    'excellent', 'perfect', 'fantastic', 'amazing', 'great', 'superb',
    'outstanding', 'wonderful', 'brilliant', 'impressive', 'stunning',
    'beautiful', 'crisp', 'rich', 'smooth', 'fast', 'clear', 'accurate',
    'intuitive', 'seamless', 'flawless', 'responsive', 'premium', 'vibrant',
    'immersive', 'comfortable', 'easy', 'well', 'love', 'delight', 'handy',
    'awesome', 'durable', 'stable', 'consistent', 'remarkable', 'fantastic',
    'sleek', 'sharp', 'enjoy', 'satisfied', 'recommend', 'happy', 'pleased',
    'good', 'best', 'top', 'quality',
}

NEGATIVE_WORDS = {
    'bad', 'poor', 'worst', 'terrible', 'horrible', 'awful', 'disappointing',
    'broken', 'failed', 'problem', 'issue', 'defective', 'slow', 'lag',
    'blur', 'noisy', 'fuzzy', 'dim', 'dull', 'cheap', 'flimsy', 'unstable',
    'difficult', 'complicated', 'confusing', 'overheating', 'crash', 'freezes',
    'glitch', 'error', 'unable', 'limited', 'useless', 'waste', 'costly',
    'expensive', 'unreliable', 'inferior', 'mediocre', 'lacking', 'worse',
    'avoid', 'regret', 'return', 'refund', 'broken', 'dead', 'defect',
}

NEUTRAL_QUALIFIERS = {
    'could', 'slightly', 'minor', 'though', 'however', 'but',
    'somewhat', 'average', 'ok', 'okay', 'decent', 'acceptable',
}

def sentiment_score(text: str) -> int:
    """
    Compute a sentiment score for cleaned text.
    Returns an integer from roughly -5 to +5.
    """
    tokens  = set(text.split())
    pos_cnt = len(tokens & POSITIVE_WORDS)
    neg_cnt = len(tokens & NEGATIVE_WORDS)
    neu_mod = len(tokens & NEUTRAL_QUALIFIERS)
    return pos_cnt - neg_cnt - (neu_mod // 2)

def score_to_label(score: int) -> int:
    """Map raw score to 3-class label (0=Negative, 1=Neutral, 2=Positive)."""
    if score >= 1:
        return 2   # Positive
    elif score == 0:
        return 1   # Neutral
    else:
        return 0   # Negative

df_raw['sentiment_score'] = df_raw['clean_text'].apply(sentiment_score)
df_raw['label']           = df_raw['sentiment_score'].apply(score_to_label)

# Verify we have all 3 classes
print("Label distribution:")
print(df_raw['label'].value_counts().rename({0:'Negative', 1:'Neutral', 2:'Positive'}))
assert set(df_raw['label'].unique()) == {0, 1, 2}, "❌ Missing label classes!"
print("\n✅ Labels created successfully!")


In [ ]:
# ─── 3.2  Visualise class distribution ───────────────────────────────────────
label_names = {0: 'Negative\n(1-2 ★)', 1: 'Neutral\n(3 ★)', 2: 'Positive\n(4-5 ★)'}
counts = df_raw['label'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
axes[0].bar([label_names[i] for i in counts.index], counts.values,
            color=['#e74c3c', '#f39c12', '#27ae60'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Class Distribution (Bar Chart)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Sentiment Class')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts.values,
            labels=[label_names[i] for i in counts.index],
            autopct='%1.1f%%', startangle=140,
            colors=['#e74c3c', '#f39c12', '#27ae60'],
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Distribution (Pie Chart)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Class distribution plot saved.")


In [ ]:
# ─── 3.3  Show sample reviews per class ──────────────────────────────────────
print("=" * 70)
for lbl, name in [(0, 'NEGATIVE'), (1, 'NEUTRAL'), (2, 'POSITIVE')]:
    print(f"\n--- {name} (label={lbl}) ---")
    samples = df_raw[df_raw['label'] == lbl]['Review Text'].head(3).tolist()
    for i, s in enumerate(samples, 1):
        print(f"  [{i}] {s[:120]}...")
print("=" * 70)


## 🔢 Section 4: Text Vectorization (Word Embeddings)

We use **Keras Tokenizer** to convert text to integer sequences and prepare embedding matrices.

**Two embedding strategies:**
1. **Trainable Embedding Layer** – randomly initialized, learned during training
2. **Pre-trained GloVe 100d** – loaded if available (`glove.6B.100d.txt`)

> If GloVe vectors are not found locally the notebook gracefully falls back to trainable embeddings.

In [ ]:
# ─── 4.1  Train / Validation / Test split ────────────────────────────────────
texts  = df_raw['clean_text'].tolist()
labels = df_raw['label'].tolist()

X_temp, X_test, y_temp, y_test = train_test_split(
    texts, labels, test_size=0.15, random_state=SEED, stratify=labels)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.15, random_state=SEED, stratify=y_temp)

print(f"Training   : {len(X_train)} samples")
print(f"Validation : {len(X_val)} samples")
print(f"Test       : {len(X_test)} samples")


In [ ]:
# ─── 4.2  Tokenize text ───────────────────────────────────────────────────────
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq   = tokenizer.texts_to_sequences(X_val)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

# Pad / truncate to MAX_LEN
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_val_pad   = pad_sequences(X_val_seq,   maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_LEN, padding='post', truncating='post')

# Convert labels to one-hot
y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_val_cat   = to_categorical(y_val,   NUM_CLASSES)
y_test_cat  = to_categorical(y_test,  NUM_CLASSES)

vocab_size = min(len(tokenizer.word_index) + 1, MAX_WORDS)
print(f"Vocabulary size   : {vocab_size}")
print(f"X_train shape     : {X_train_pad.shape}")
print(f"y_train shape     : {y_train_cat.shape}")


In [ ]:
# ─── 4.3  Load GloVe embeddings (optional, falls back to trainable) ───────────
GLOVE_PATH  = 'glove.6B.100d.txt'
glove_found = os.path.isfile(GLOVE_PATH)

embedding_matrix = np.zeros((vocab_size, EMBED_DIM))

if glove_found:
    print("✅ GloVe file found — loading vectors...")
    glove_index = {}
    with open(GLOVE_PATH, encoding='utf-8') as f:
        for line in f:
            parts = line.split()
            word  = parts[0]
            vec   = np.asarray(parts[1:], dtype='float32')
            glove_index[word] = vec

    hit, miss = 0, 0
    for word, idx in tokenizer.word_index.items():
        if idx < vocab_size:
            vec = glove_index.get(word)
            if vec is not None:
                embedding_matrix[idx] = vec
                hit += 1
            else:
                miss += 1

    print(f"GloVe hits  : {hit}")
    print(f"GloVe misses: {miss}")
    print(f"Coverage    : {hit/(hit+miss)*100:.1f}%")
else:
    print("ℹ️  GloVe file NOT found — using trainable embeddings (random init).")
    print("   To use GloVe, download glove.6B.zip from https://nlp.stanford.edu/projects/glove/")
    print("   and place glove.6B.100d.txt in the project directory.")

TRAINABLE_EMBEDDINGS = not glove_found
print(f"\nTrainable embeddings: {TRAINABLE_EMBEDDINGS}")


## 🧠 Section 5: Model 1 — Dense Neural Network (Baseline)

**Architecture:** `Embedding → GlobalAveragePooling1D → Dense(128) → Dropout → Dense(64) → Dropout → Dense(3, softmax)`

The simplest neural baseline — treats the document as a bag of embedded words (averaged).

In [ ]:
# ─── 5.1  Build Dense NN ─────────────────────────────────────────────────────
def build_dense_model(vocab_size, embed_dim, max_len, num_classes,
                       embedding_matrix=None, trainable=True):
    """Dense NN baseline using Embedding + GlobalAveragePooling."""
    inp = Input(shape=(max_len,), name='input')

    if embedding_matrix is not None and not trainable:
        emb = layers.Embedding(vocab_size, embed_dim,
                               weights=[embedding_matrix],
                               trainable=False, name='glove_embedding')(inp)
    else:
        emb = layers.Embedding(vocab_size, embed_dim, name='trainable_embedding')(inp)

    x = layers.GlobalAveragePooling1D(name='global_avg_pool')(emb)
    x = layers.Dense(128, activation='relu', name='dense_1')(x)
    x = layers.Dropout(0.4, name='dropout_1')(x)
    x = layers.Dense(64,  activation='relu', name='dense_2')(x)
    x = layers.Dropout(0.3, name='dropout_2')(x)
    out = layers.Dense(num_classes, activation='softmax', name='output')(x)

    model = Model(inputs=inp, outputs=out, name='Dense_NN_Baseline')
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

dense_model = build_dense_model(
    vocab_size, EMBED_DIM, MAX_LEN, NUM_CLASSES,
    embedding_matrix=(None if TRAINABLE_EMBEDDINGS else embedding_matrix),
    trainable=TRAINABLE_EMBEDDINGS
)
dense_model.summary()


In [ ]:
# ─── 5.2  Train Dense NN ─────────────────────────────────────────────────────
callbacks_dense = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=0),
]

history_dense = dense_model.fit(
    X_train_pad, y_train_cat,
    validation_data=(X_val_pad, y_val_cat),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_dense,
    verbose=1,
)
print("\n✅ Dense NN training complete!")


## 🔁 Section 6: Model 2 — LSTM / GRU Recurrent Neural Network

**Architecture:** `Embedding → SpatialDropout1D → LSTM(128) → Dropout → Dense(64) → Dense(3, softmax)`

LSTMs capture **sequential dependencies** in text — the order of words matters.

In [ ]:
# ─── 6.1  Build LSTM model ───────────────────────────────────────────────────
def build_lstm_model(vocab_size, embed_dim, max_len, num_classes,
                      embedding_matrix=None, trainable=True):
    """LSTM-based text classifier."""
    inp = Input(shape=(max_len,), name='input')

    if embedding_matrix is not None and not trainable:
        emb = layers.Embedding(vocab_size, embed_dim,
                               weights=[embedding_matrix],
                               trainable=False, name='glove_embedding')(inp)
    else:
        emb = layers.Embedding(vocab_size, embed_dim, name='trainable_embedding')(inp)

    x = layers.SpatialDropout1D(0.2, name='spatial_dropout')(emb)
    x = layers.LSTM(128, dropout=0.3, recurrent_dropout=0.2, name='lstm_1')(x)
    x = layers.Dense(64, activation='relu', name='dense_1')(x)
    x = layers.Dropout(0.3, name='dropout_1')(x)
    out = layers.Dense(num_classes, activation='softmax', name='output')(x)

    model = Model(inputs=inp, outputs=out, name='LSTM_RNN')
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

lstm_model = build_lstm_model(
    vocab_size, EMBED_DIM, MAX_LEN, NUM_CLASSES,
    embedding_matrix=(None if TRAINABLE_EMBEDDINGS else embedding_matrix),
    trainable=TRAINABLE_EMBEDDINGS
)
lstm_model.summary()


In [ ]:
# ─── 6.2  Train LSTM ─────────────────────────────────────────────────────────
callbacks_lstm = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=0),
]

history_lstm = lstm_model.fit(
    X_train_pad, y_train_cat,
    validation_data=(X_val_pad, y_val_cat),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_lstm,
    verbose=1,
)
print("\n✅ LSTM training complete!")


## 🔭 Section 7: Model 3 — 1D CNN Text Classifier

**Architecture:** `Embedding → Conv1D(128, kernel=5) → GlobalMaxPool1D → Dense(64) → Dropout → Dense(3, softmax)`

CNNs capture **local n-gram patterns** — very efficient for text classification.

In [ ]:
# ─── 7.1  Build 1D-CNN model ──────────────────────────────────────────────────
def build_cnn_model(vocab_size, embed_dim, max_len, num_classes,
                     embedding_matrix=None, trainable=True):
    """1D CNN text classifier."""
    inp = Input(shape=(max_len,), name='input')

    if embedding_matrix is not None and not trainable:
        emb = layers.Embedding(vocab_size, embed_dim,
                               weights=[embedding_matrix],
                               trainable=False, name='glove_embedding')(inp)
    else:
        emb = layers.Embedding(vocab_size, embed_dim, name='trainable_embedding')(inp)

    x = layers.SpatialDropout1D(0.2, name='spatial_dropout')(emb)

    # Multi-scale convolutions (bigram, trigram, 4-gram)
    conv3 = layers.Conv1D(128, 3, activation='relu', padding='same', name='conv_3')(x)
    conv4 = layers.Conv1D(128, 4, activation='relu', padding='same', name='conv_4')(x)
    conv5 = layers.Conv1D(128, 5, activation='relu', padding='same', name='conv_5')(x)

    pool3 = layers.GlobalMaxPooling1D(name='pool_3')(conv3)
    pool4 = layers.GlobalMaxPooling1D(name='pool_4')(conv4)
    pool5 = layers.GlobalMaxPooling1D(name='pool_5')(conv5)

    x = layers.Concatenate(name='concatenate')([pool3, pool4, pool5])
    x = layers.Dense(128, activation='relu', name='dense_1')(x)
    x = layers.Dropout(0.4, name='dropout_1')(x)
    x = layers.Dense(64,  activation='relu', name='dense_2')(x)
    x = layers.Dropout(0.3, name='dropout_2')(x)
    out = layers.Dense(num_classes, activation='softmax', name='output')(x)

    model = Model(inputs=inp, outputs=out, name='CNN_1D_TextClassifier')
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

cnn_model = build_cnn_model(
    vocab_size, EMBED_DIM, MAX_LEN, NUM_CLASSES,
    embedding_matrix=(None if TRAINABLE_EMBEDDINGS else embedding_matrix),
    trainable=TRAINABLE_EMBEDDINGS
)
cnn_model.summary()


In [ ]:
# ─── 7.2  Train 1D-CNN ────────────────────────────────────────────────────────
callbacks_cnn = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=0),
]

history_cnn = cnn_model.fit(
    X_train_pad, y_train_cat,
    validation_data=(X_val_pad, y_val_cat),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_cnn,
    verbose=1,
)
print("\n✅ 1D-CNN training complete!")


## 🏆 Section 8: Model 4 — Bidirectional LSTM with Attention (Advanced)

**Architecture:** `Embedding → SpatialDropout → BiLSTM(128) → Attention → Dense(64) → Dropout → Dense(3, softmax)`

**Bidirectional LSTM** reads text in both forward and backward directions, capturing richer context.  
**Bahdanau-style Attention** lets the model focus on the most relevant words for classification.

In [ ]:
# ─── 8.1  Custom Attention Layer ────────────────────────────────────────────
class BahdanauAttention(layers.Layer):
    """
    Bahdanau (additive) Attention mechanism.
    Computes a context vector as a weighted sum of LSTM hidden states.
    """
    def __init__(self, units=64, **kwargs):
        super().__init__(**kwargs)
        self.W = layers.Dense(units)
        self.V = layers.Dense(1)

    def call(self, query):
        # query: (batch, timesteps, hidden_dim)
        score = self.V(tf.nn.tanh(self.W(query)))   # (batch, timesteps, 1)
        weights = tf.nn.softmax(score, axis=1)        # attention weights
        context = weights * query                     # (batch, timesteps, hidden_dim)
        context = tf.reduce_sum(context, axis=1)      # (batch, hidden_dim)
        return context, weights


# ─── 8.2  Build BiLSTM + Attention model ─────────────────────────────────────
def build_bilstm_attention_model(vocab_size, embed_dim, max_len, num_classes,
                                  embedding_matrix=None, trainable=True):
    """Bidirectional LSTM with Bahdanau Attention classifier."""
    inp = Input(shape=(max_len,), name='input')

    if embedding_matrix is not None and not trainable:
        emb = layers.Embedding(vocab_size, embed_dim,
                               weights=[embedding_matrix],
                               trainable=False, name='glove_embedding')(inp)
    else:
        emb = layers.Embedding(vocab_size, embed_dim, name='trainable_embedding')(inp)

    x = layers.SpatialDropout1D(0.2, name='spatial_dropout')(emb)

    # Bidirectional LSTM — return_sequences=True to get all hidden states
    x = layers.Bidirectional(
            layers.LSTM(128, dropout=0.3, recurrent_dropout=0.2,
                        return_sequences=True),
            name='bilstm')(x)

    # Attention
    context, attn_weights = BahdanauAttention(units=64, name='attention')(x)

    x = layers.Dense(64, activation='relu', name='dense_1')(context)
    x = layers.Dropout(0.3, name='dropout_1')(x)
    out = layers.Dense(num_classes, activation='softmax', name='output')(x)

    model = Model(inputs=inp, outputs=out, name='BiLSTM_Attention')
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

bilstm_model = build_bilstm_attention_model(
    vocab_size, EMBED_DIM, MAX_LEN, NUM_CLASSES,
    embedding_matrix=(None if TRAINABLE_EMBEDDINGS else embedding_matrix),
    trainable=TRAINABLE_EMBEDDINGS
)
bilstm_model.summary()


In [ ]:
# ─── 8.3  Train BiLSTM + Attention ───────────────────────────────────────────
callbacks_bilstm = [
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=0),
]

history_bilstm = bilstm_model.fit(
    X_train_pad, y_train_cat,
    validation_data=(X_val_pad, y_val_cat),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_bilstm,
    verbose=1,
)
print("\n✅ BiLSTM + Attention training complete!")


## 📈 Section 9: Model Evaluation

Compare all 4 models on the **held-out test set** using:
- **Accuracy, Precision, Recall, F1-Score** (per-class and macro-average)
- **Confusion Matrices** (heatmaps)
- **Training vs Validation Loss and Accuracy plots**

In [ ]:
# ─── 9.1  Utility: plot training history ─────────────────────────────────────
def plot_training_history(history, model_name):
    """Plot loss and accuracy curves for training and validation."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(f'{model_name} — Training History', fontsize=14, fontweight='bold')

    # Loss
    axes[0].plot(history.history['loss'],     label='Train Loss',     color='royalblue')
    axes[0].plot(history.history['val_loss'], label='Val Loss',       color='tomato', linestyle='--')
    axes[0].set_title('Loss (Cross-Entropy)')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Accuracy
    axes[1].plot(history.history['accuracy'],     label='Train Accuracy', color='royalblue')
    axes[1].plot(history.history['val_accuracy'], label='Val Accuracy',   color='tomato', linestyle='--')
    axes[1].set_title('Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    fname = f'history_{model_name.lower().replace(" ", "_")}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✅ History plot saved as {fname}")


# Plot for all models
plot_training_history(history_dense,  'Dense NN')
plot_training_history(history_lstm,   'LSTM')
plot_training_history(history_cnn,    '1D-CNN')
plot_training_history(history_bilstm, 'BiLSTM+Attention')


In [ ]:
# ─── 9.2  Utility: evaluate a model and print report ─────────────────────────
label_names = ['Negative', 'Neutral', 'Positive']

def evaluate_model(model, X_test_pad, y_test, model_name):
    """
    Evaluate a trained model on the test set.
    Returns a dict of metrics and predicted labels.
    """
    y_pred_proba = model.predict(X_test_pad, verbose=0)
    y_pred       = np.argmax(y_pred_proba, axis=1)
    y_true       = np.array(y_test)

    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1   = f1_score(y_true, y_pred, average='macro', zero_division=0)

    print(f"\n{'='*55}")
    print(f"  {model_name} — Test Set Results")
    print(f"{'='*55}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f} (macro)")
    print(f"  Recall    : {rec:.4f} (macro)")
    print(f"  F1-Score  : {f1:.4f} (macro)")
    print(f"{'='*55}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))

    return {'name': model_name, 'acc': acc, 'prec': prec, 'rec': rec, 'f1': f1, 'y_pred': y_pred}


# Evaluate all models
results_dense  = evaluate_model(dense_model,  X_test_pad, y_test, 'Dense NN')
results_lstm   = evaluate_model(lstm_model,   X_test_pad, y_test, 'LSTM')
results_cnn    = evaluate_model(cnn_model,    X_test_pad, y_test, '1D-CNN')
results_bilstm = evaluate_model(bilstm_model, X_test_pad, y_test, 'BiLSTM+Attention')


In [ ]:
# ─── 9.3  Confusion Matrices ──────────────────────────────────────────────────
def plot_confusion_matrix(y_true, y_pred, model_name):
    """Plot a heatmap confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_names, yticklabels=label_names,
                linewidths=0.5, linecolor='white')
    plt.title(f'Confusion Matrix — {model_name}', fontsize=12, fontweight='bold')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    fname = f'cm_{model_name.lower().replace(" ", "_").replace("+", "")}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✅ Confusion matrix saved as {fname}")

y_true = np.array(y_test)
plot_confusion_matrix(y_true, results_dense['y_pred'],  'Dense NN')
plot_confusion_matrix(y_true, results_lstm['y_pred'],   'LSTM')
plot_confusion_matrix(y_true, results_cnn['y_pred'],    '1D-CNN')
plot_confusion_matrix(y_true, results_bilstm['y_pred'], 'BiLSTM+Attention')


In [ ]:
# ─── 9.4  Side-by-side model comparison table ────────────────────────────────
all_results = [results_dense, results_lstm, results_cnn, results_bilstm]

comparison_df = pd.DataFrame([{
    'Model'    : r['name'],
    'Accuracy' : round(r['acc'],  4),
    'Precision': round(r['prec'], 4),
    'Recall'   : round(r['rec'],  4),
    'F1-Score' : round(r['f1'],   4),
} for r in all_results])

comparison_df = comparison_df.set_index('Model')
print("\n📊 Model Comparison Table:")
print(comparison_df.to_string())

# Bar chart comparison
comparison_df.plot(kind='bar', figsize=(12, 5), colormap='Set2',
                   edgecolor='white', linewidth=1.2)
plt.title('Model Comparison — All Metrics', fontsize=13, fontweight='bold')
plt.ylabel('Score')
plt.xticks(rotation=20, ha='right')
plt.ylim(0, 1.05)
plt.legend(loc='lower right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Comparison chart saved.")


## ⚙️ Section 10: Model Optimization (Hyperparameter Tuning)

We manually grid-search over the most impactful hyperparameters for the **best-performing model** (BiLSTM+Attention):

| Hyperparameter | Values Tried |
|---------------|-------------|
| Learning rate | 1e-3, 5e-4, 1e-4 |
| Batch size    | 32, 64, 128 |
| Dropout rate  | 0.2, 0.3, 0.4 |
| LSTM units    | 64, 128 |
| Embedding dim | 100 |

In [ ]:
# ─── 10.1  Manual grid search over key hyperparameters ───────────────────────
import itertools

param_grid = {
    'lr'      : [1e-3, 5e-4],
    'dropout' : [0.2, 0.4],
    'lstm_units': [64, 128],
    'batch'   : [32, 64],
}

# Create all combinations
keys   = list(param_grid.keys())
values = list(param_grid.values())
combos = list(itertools.product(*values))

# Limit to 6 combinations to keep runtime manageable
np.random.shuffle(combos)
combos = combos[:6]

print(f"Testing {len(combos)} hyperparameter combinations...\n")

best_f1      = 0.0
best_params  = {}
tuning_log   = []

for combo in combos:
    params = dict(zip(keys, combo))
    print(f"Trying: {params}")

    # Build a compact BiLSTM for speed
    inp = Input(shape=(MAX_LEN,))
    emb = layers.Embedding(vocab_size, EMBED_DIM)(inp)
    x   = layers.SpatialDropout1D(params['dropout'])(emb)
    x   = layers.Bidirectional(layers.LSTM(params['lstm_units'], dropout=params['dropout']))(x)
    x   = layers.Dense(32, activation='relu')(x)
    x   = layers.Dropout(params['dropout'])(x)
    out = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    m   = Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(lr=params['lr']),
              loss='categorical_crossentropy', metrics=['accuracy'])

    cb = [EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)]
    m.fit(X_train_pad, y_train_cat,
          validation_data=(X_val_pad, y_val_cat),
          epochs=5, batch_size=params['batch'],
          callbacks=cb, verbose=0)

    y_pred = np.argmax(m.predict(X_test_pad, verbose=0), axis=1)
    f1_val = f1_score(np.array(y_test), y_pred, average='macro', zero_division=0)
    acc_val = accuracy_score(np.array(y_test), y_pred)
    print(f"  → Val F1 = {f1_val:.4f}, Acc = {acc_val:.4f}\n")
    tuning_log.append({**params, 'f1': f1_val, 'acc': acc_val})

    if f1_val > best_f1:
        best_f1     = f1_val
        best_params = params

print("=" * 55)
print(f"Best Hyperparameters  : {best_params}")
print(f"Best Validation F1    : {best_f1:.4f}")
print("=" * 55)

tuning_df = pd.DataFrame(tuning_log).round(4)
print("\nFull tuning results:")
print(tuning_df.to_string(index=False))


## 🔍 Section 11: Error Analysis

Examine which reviews the **best model (BiLSTM+Attention)** got wrong.  
Understanding misclassifications reveals dataset noise, model biases, or ambiguous reviews.

In [ ]:
# ─── 11.1  Collect misclassified samples ─────────────────────────────────────
y_pred_best = results_bilstm['y_pred']
y_true_arr  = np.array(y_test)

# DataFrame of test examples with predictions
test_df = pd.DataFrame({
    'Review Text'    : X_test,
    'True Label'     : y_true_arr,
    'Predicted Label': y_pred_best,
    'True Name'      : [label_names[l] for l in y_true_arr],
    'Pred Name'      : [label_names[l] for l in y_pred_best],
})

mis_df = test_df[test_df['True Label'] != test_df['Predicted Label']].copy()
print(f"Total misclassified : {len(mis_df)} / {len(test_df)} ({len(mis_df)/len(test_df)*100:.1f}%)")
print(f"Correctly classified: {len(test_df)-len(mis_df)} / {len(test_df)} ({(1-len(mis_df)/len(test_df))*100:.1f}%)")


In [ ]:
# ─── 11.2  Error distribution by class pair ───────────────────────────────────
error_pairs = mis_df.groupby(['True Name', 'Pred Name']).size().reset_index(name='Count')
error_pairs = error_pairs.sort_values('Count', ascending=False)
print("\nMisclassification breakdown:")
print(error_pairs.to_string(index=False))

# Heatmap of confusion pair counts
pivot = error_pairs.pivot(index='True Name', columns='Pred Name', values='Count').fillna(0)
plt.figure(figsize=(6, 4))
sns.heatmap(pivot.astype(int), annot=True, fmt='d', cmap='Reds',
            linewidths=0.5, linecolor='white')
plt.title('Misclassification Heatmap — BiLSTM+Attention', fontsize=12, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('error_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ─── 11.3  Display sample misclassified reviews ───────────────────────────────
print("\n=== SAMPLE MISCLASSIFIED REVIEWS ===")
for _, row in mis_df.sample(min(10, len(mis_df)), random_state=SEED).iterrows():
    print(f"\n  True: {row['True Name']:8s} | Predicted: {row['Pred Name']}")
    print(f"  Review: {row['Review Text'][:140]}")
    print("  " + "-"*70)


## 🌟 Section 12 (Bonus): LIME Word-Importance Explainability

**LIME (Local Interpretable Model-Agnostic Explanations)** explains individual predictions by identifying which words most influenced the model's decision.

> Install LIME if not present: `pip install lime`

In [ ]:
# ─── 12.1  LIME setup ─────────────────────────────────────────────────────────
try:
    from lime.lime_text import LimeTextExplainer
    LIME_AVAILABLE = True
    print("✅ LIME is available!")
except ImportError:
    LIME_AVAILABLE = False
    print("ℹ️  LIME not installed. Run: pip install lime")
    print("   Skipping LIME explainability section.")


In [ ]:
# ─── 12.2  LIME prediction function + explanations ────────────────────────────
if LIME_AVAILABLE:

    def predict_proba_fn(texts):
        """Wrapper so LIME can call our BiLSTM model."""
        seqs = tokenizer.texts_to_sequences(texts)
        pads = pad_sequences(seqs, maxlen=MAX_LEN, padding='post', truncating='post')
        return bilstm_model.predict(pads, verbose=0)

    explainer = LimeTextExplainer(class_names=label_names, random_state=SEED)

    # Pick 3 diverse test samples
    sample_indices = []
    for target_class in [0, 1, 2]:
        idxs = np.where(y_true_arr == target_class)[0]
        if len(idxs) > 0:
            sample_indices.append(int(idxs[0]))

    for idx in sample_indices:
        review_text = X_test[idx]
        true_label  = label_names[y_true_arr[idx]]
        pred_label  = label_names[y_pred_best[idx]]

        print(f"\n{'='*65}")
        print(f"Review (idx={idx})")
        print(f"  True: {true_label} | Predicted: {pred_label}")
        print(f"  Text: {review_text[:120]}...")

        exp = explainer.explain_instance(
            review_text, predict_proba_fn,
            num_features=10, num_samples=200,
            labels=[y_true_arr[idx]]
        )

        # Save HTML explanation
        html_path = f'lime_explanation_{true_label.lower()}.html'
        exp.save_to_file(html_path)
        print(f"  ✅ LIME explanation saved: {html_path}")

        # Plot inline
        fig = exp.as_pyplot_figure(label=y_true_arr[idx])
        fig.suptitle(f'LIME — True: {true_label} | Pred: {pred_label}',
                     fontsize=11, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'lime_plot_{true_label.lower()}.png', dpi=150, bbox_inches='tight')
        plt.show()

else:
    print("Skipped — LIME not installed.")


## ✅ Project Summary

| Section | Task | Status |
|---------|------|--------|
| 0 | Setup & Imports | ✅ |
| 1 | Data Loading & EDA | ✅ |
| 2 | Text Cleaning & Preprocessing | ✅ |
| 3 | Sentiment Labeling (3-class) | ✅ |
| 4 | Text Vectorization (GloVe/Trainable) | ✅ |
| 5 | Dense NN Baseline | ✅ |
| 6 | LSTM / GRU RNN | ✅ |
| 7 | 1D CNN Text Classifier | ✅ |
| 8 | BiLSTM + Bahdanau Attention | ✅ |
| 9 | Evaluation (Metrics, CM, History plots) | ✅ |
| 10 | Hyperparameter Optimization | ✅ |
| 11 | Error Analysis | ✅ |
| 12 | LIME Explainability (Bonus) | ✅ |

---
### Key Design Decisions
- **No traditional ML** (Naive Bayes, SVM, etc.) — only Neural Networks used
- **No pre-built sentiment APIs** (TextBlob, VADER, etc.) — custom keyword scorer
- **GloVe 100d** used if available; graceful fallback to trainable embeddings
- **3-class labeling**: Negative (0), Neutral (1), Positive (2)
- **Evaluation metrics**: Accuracy, Precision, Recall, F1 (macro-averaged)